# Querying External Data Sources using GA4GH VRS

## Setup

In [ ]:
!pip install vrs_anvil_toolkit

## Converting a variant to VRS

The following code will demonstrate how to convert an HGVS sequenc. For more information on how to use VRS in Python, see the [Getting Started](https://github.com/ga4gh/vrs-python/tree/main/notebooks/getting_started) notebooks in the [vrs-python](https://github.com/ga4gh/vrs-python) repo.

In [6]:
from ga4gh.vrs.extras.translator import AlleleTranslator
from ga4gh.vrs.dataproxy import create_dataproxy

In [7]:
# provide your variant id
variant_id = "NC_000007.13:g.87138645A>G"

# setup your remote reference data
seqrepo_rest_service_url = "seqrepo+https://services.genomicmedlab.org/seqrepo"
seqrepo_dataproxy = create_dataproxy(uri=seqrepo_rest_service_url)

# translate the variant to a VRS ID using the AlleleTranslator
allele_translator = AlleleTranslator(seqrepo_dataproxy)
allele = allele_translator.translate_from(variant_id, "hgvs")
vrs_id = allele.id

print("VRS ID:", vrs_id)

VRS ID: ga4gh:VA.uwlGZbX2QG-Ubf-AxKT9FxdL9dVt1KzC


## Querying MetaKB for VCF variants

In [9]:
from collections import defaultdict
from vrs_anvil import Manifest, query_metakb
from vrs_anvil.annotator import annotate_all, MATCHES, TOTAL

import os
import json
import pysam
import yaml

In [8]:
# get VCF
# TODO: replace it with existing stuff
vcf_path = "/Users/wongq/data/1kGP/1kGP_high_coverage_Illumina.chr1.11700000_to_12000000.filtered.SNV_INDEL_SV_phased_panel.vcf"
# setup manifest with your VCF
base_manifest_yaml = yaml.safe_load(open("/Users/wongq/projects/vrs_anvil_toolkit/tests/fixtures/manifest.yaml", "r"))
base_manifest_yaml["vcf_files"] = [vcf_path]
# base_manifest_yaml["metakb_directory"] = "/Users/wongq/projects/vrs_anvil_toolkit/tests/fixtures/metakb"
base_manifest_yaml["metakb_directory"] = "metakb"
manifest = Manifest.model_validate(base_manifest_yaml)


# call vrs_bulk to create metrics
metrics_path = annotate_all(manifest=manifest, max_errors=0)
print(metrics_path)


NameError: name 'yaml' is not defined

In [12]:
# read metrics files and query metakb for hits
# pull in things from civic

with open(metrics_path, "r") as file:
    metrics = yaml.safe_load(file)

print("metrics file:")
print(json.dumps(metrics, indent=2))

metrics file:
{
  "total": {
    "elapsed_time": 8.321766138076782,
    "end_time": 1741973158.759546,
    "errors": 0,
    "start_time": 1741973150.43778,
    "successes": 9026,
    "timestamp_str": null
  },
  "work/1kGP_high_coverage_Illumina.chr1.11700000_to_12000000.filtered.SNV_INDEL_SV_phased_panel.vcf": {
    "elapsed_time": 7.307703971862793,
    "end_time": 1741973157.749742,
    "errors": {},
    "line_count": 9041,
    "matches": {
      "ga4gh:VA.SOEVGpU16hxYQtJNeRyfq0V-B0rSOGK-": {
        "fmt": "gnomad",
        "var": "chr1-11796321-G-A"
      }
    },
    "metakb_hits": 1,
    "start_time": 1741973150.442038,
    "status": "finished",
    "successes": 9026
  }
}


In [22]:
# store the matches for each file dict
matches_per_file = {}

for file_path, metrics_dict in metrics.items():
    if file_path == TOTAL:
        continue
    
    # load original vcf
    original_path = os.readlink(file_path)
    print(original_path)
    
    # show matches
    if MATCHES in metrics_dict:
        num_matches = len(metrics_dict[MATCHES])
        matches_dict = metrics_dict[MATCHES]
        print(f"{num_matches} match{"" if num_matches == 1 else "es"} with metaKB found")
        matches_per_file.update({original_path: metrics_dict[MATCHES]})
    else:
        print("no matches to metaKB found.")
    
    print()


/Users/wongq/data/1kGP/1kGP_high_coverage_Illumina.chr1.11700000_to_12000000.filtered.SNV_INDEL_SV_phased_panel.vcf
total samples in VCF: 3202
1 match with metaKB found



In [34]:
variant_evidence_dict = {}

# parse variant matches to save metakb study data
for file_path, matches in matches_per_file.items():
    vcf_reader = pysam.VariantFile(file_path)

    print("matches for...")
    print(file_path)

    # for each variant per file
    for allele_id, allele_info in matches.items():
        sample_dict = defaultdict(list)

        print("Variant:", allele_id)
        print("Source ID:", allele_info["var"])

        # get study id associated with vrs allele id
        metakb_response = query_metakb(allele_id, log=True)
        if metakb_response is None:
            print(f"no metakb hit for allele {allele_id}\n")
        else:
            study_ids = metakb_response["study_ids"]

            assert study_ids == [
                study["id"] for study in metakb_response["studies"]
            ], "study ids in wrong order to add variant types"
            variant_types = [
                study["qualifiers"]["alleleOrigin"]
                for study in metakb_response["studies"]
            ]

            studies = {}
            for study in metakb_response["studies"]:
                print(f"\t{study['type']} ({study['id']}): {study['description']}")
            print()

            variant_evidence_dict[allele_id] = {
                "file": file_path,
                "format": allele_info["fmt"],
                "source_id": allele_info["var"],
                "studies": metakb_response["studies"]

            }
        # TODO: create caf dict?
        # caf_dict = create_caf_dict(
        #     allele_id,
        #     gnomad_expr,
        #     focus_allele,
        #     focus_allele_count,
        #     locus_allele_count,
        #     ancillary_results,
        # )

    print("~~~~~~~~~~~~~\n")


print("saved full results to variant_evidence_dict!")


matches for...
/Users/wongq/data/1kGP/1kGP_high_coverage_Illumina.chr1.11700000_to_12000000.filtered.SNV_INDEL_SV_phased_panel.vcf
Variant: ga4gh:VA.SOEVGpU16hxYQtJNeRyfq0V-B0rSOGK-
Source ID: chr1-11796321-G-A
	VariantTherapeuticResponseStudy (civic.eid:669): The MTHFR C667T variant was associated with significantly lower relapse-free survival and overall survival in stomach cancer patients treated with 5-Fluorouracil-based therapies. 116 Chinese patients with histologically confirmed gastric cancer were used in this study, and all patients had radical surgery before treatment.
	VariantTherapeuticResponseStudy (civic.eid:1757): Patients with the wild type (C/C) MTHFR gene are 2.91 times (95% CI: [1.23, 6.89]) more likely to have a positive response to neoadjuvant CRT and  3.25 times more likely not to experience relapse (95% CI: [1.37, 7.72]) than patients with the heterozygous  MTHFR [rs1801133 (C>T)] mutation or  the homzygous (T/T).

~~~~~~~~~~~~~

saved full results to variant_evi

In [42]:
# get a list of links by evidence type
print("List of CIVIC links (if matches to studies)")
for allele_id, allele_dict in variant_evidence_dict.items():
    for study in allele_dict["studies"]:
        if "eid" in study["id"]:
            study_id_num = study["id"].split()[-1]
            print(f"- https://civicdb.org/evidence/{study_id_num}/summary")

List of CIVIC links (if matches to studies)
- https://civicdb.org/evidence/civic.eid:669/summary
- https://civicdb.org/evidence/civic.eid:1757/summary


In [36]:
# these details are also pulled into the json
print(json.dumps(variant_evidence_dict, indent=2))

{
  "ga4gh:VA.SOEVGpU16hxYQtJNeRyfq0V-B0rSOGK-": {
    "file": "/Users/wongq/data/1kGP/1kGP_high_coverage_Illumina.chr1.11700000_to_12000000.filtered.SNV_INDEL_SV_phased_panel.vcf",
    "format": "gnomad",
    "source_id": "chr1-11796321-G-A",
    "studies": [
      {
        "id": "civic.eid:669",
        "description": "The MTHFR C667T variant was associated with significantly lower relapse-free survival and overall survival in stomach cancer patients treated with 5-Fluorouracil-based therapies. 116 Chinese patients with histologically confirmed gastric cancer were used in this study, and all patients had radical surgery before treatment.",
        "type": "VariantTherapeuticResponseStudy",
        "specifiedBy": {
          "id": "civic.method:2019",
          "label": "CIViC Curation SOP (2019)",
          "type": "Method",
          "isReportedIn": {
            "label": "Danos et al., 2019, Genome Med.",
            "type": "Document",
            "title": "Standard operating pro